# Programming in Java – Assignment 4 (CO4)
## Event-Driven GUI Design

**Student:** Archana V  
**Register Number:** 192521420  
**Faculty:** Ravi Pujitha  
**Academic Year:** 2026–2027  
**Course:** Programming in Java (CSA09)  
**Total:** 50 marks

This notebook follows the uploaded Assignment 4 PDF. GUI programs use Java AWT/Swing. Colab is headless, so windows cannot be shown here; compile checks are possible, but run the GUIs on a desktop Java environment for screenshots.

In [1]:
import subprocess
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
print(subprocess.run(["javac", "-version"], capture_output=True, text=True).stderr)

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)




## Task 1 – GUI Design Review (6 marks)
### 1.1 Diagnose the design
1. UI and payment/database logic are combined in one listener, making testing and maintenance difficult.
2. `==` compares String object references; use `.equals()` or dedicated listeners/actions.
3. Empty, nonnumeric, negative or invalid amounts are not validated and can cause parsing errors.
4. Database work and `Thread.sleep(3000)` occur on the event-dispatch thread (EDT), freezing the window.
5. Feedback is only printed to console, not shown to the user.
6. Receipt is not handled; it falls into the Clear branch.
7. SQL concatenation is vulnerable to injection and formatting errors.
8. Connection/statement are not closed and closing the window is not handled explicitly.
9. Checked exceptions such as `SQLException` and `InterruptedException` are not handled.

### 1.2 Repair the design
**FeeWindow:** gathers input, invokes service, displays status, handles window events. **FeeService:** validates payment and processes business logic. **FeeDAO:** performs database operations with prepared statements.

Class flow: `FeeWindow → FeeService → FeeDAO → Database`

Handler pseudocode:
```text
Pay click: read ID and amount; call service; display success/error.
eceipt click: request/display receipt.\nClear click: clear fields.\n```

### 1.3 Design defence (under 100 words)
Separating the interface from the service keeps the window responsible for input and output only. FeeService can be tested independently with valid, invalid and boundary values without opening a window. Database code can change without rewriting button handlers. Each handler is smaller and easier to understand, errors can be reported consistently, and the design is easier to extend with receipt generation or additional payment methods.

### Task 1 – Alternative and complexity
A dedicated listener for each button or a shared Swing `Action` avoids an `if–else` chain of up to k command comparisons. Dispatch selection is approximately **O(1)**, compared with worst-case **O(k)** comparisons for k buttons. The separate design stores **O(k)** listener/action objects. Shared `Action` objects can centralize labels, enabled state and behavior.

## Task 2 – Hospital Outpatient Appointment Desk (8 marks)
### 2.1 Event table
| Source | Event object | Listener | Action |
|---|---|---|
| Department combo | `ItemEvent` | `ItemListener` | Refresh doctor choices for department |
| Doctor combo | `ItemEvent` | `ItemListener` | Update doctor selection |
| Time-slot combo | `ItemEvent` | `ItemListener` | Record selected slot |
| Book button | `ActionEvent` | `ActionListener` | Validate and book appointment |
| Patient ID field (Enter) | `ActionEvent` | `ActionListener` | Optional submit |
| Window close | `WindowEvent` | `WindowAdapter` | Confirm if booking in progress |

### 2.2 Component containment
```text
JFrame Appointment Desk (BorderLayout)
├─ NORTH: JPanel (GridLayout form)
│  ├─ JLabel + JTextField (patient ID)
│  ├─ JLabel + JComboBox (department)
│  ├─ JLabel + JComboBox (doctor)
│  └─ JLabel + JComboBox (time)
├─ CENTER: JPanel (appointment details/status)
└─ SOUTH: JPanel (FlowLayout) – Book button + status label
```

### 2.3 Handler pseudocode
```text
ON department change:
  selectedDept = department combo value
  doctorList = doctorMap.get(selectedDept)
  clear doctor combo; populate it with doctorList
ON Book:
  if patient ID blank or doctor/slot missing: show validation; return
  if bookingInProgress: return
  bookingInProgress = true
  verify slot and save booking
  display confirmation or error
  bookingInProgress = false
```

### 2.4 Event sequence and applet comparison
Frame opens: construct frame/components → register listeners/load data → `setVisible(true)` → user event triggers listener on EDT → close event invokes `windowClosing` → confirm if booking is in progress → cancel keeps window; confirm disposes it. A legacy applet lifecycle is `init() → start() → paint() (as needed) → stop() → destroy()`. A desktop frame does not follow that applet lifecycle.

### Task 2 – Alternative and complexity
If doctors are queried from the database on every department change, each change adds query and network latency; returning m doctors is **O(m)** data processing. A preloaded `Map<String,List<Doctor>>` has average **O(1)** map lookup, then **O(m)** to populate the combo. Preloading d departments with m doctors each uses about **O(d·m)** memory. Fetch-on-demand stores roughly the current list **O(m)** in client memory but repeats queries and can be slower.

## Task 3 – Campus Library Search Screen (9 marks)
### 3.1 Layout manager choices
- Outer `BorderLayout`: search row NORTH, results CENTER (grows), controls/status SOUTH.
- Search row `FlowLayout`: query field, category and Search button aligned horizontally.
- Results area `BorderLayout` (or `GridLayout` for result tiles), fills available centre space.
- Bottom button row `FlowLayout`: Prev, Next, Reserve.
- `CardLayout` may switch between results, empty and error views.

### 3.2 Panel nesting
```text
JFrame (BorderLayout)
├─ NORTH: JPanel FlowLayout – search field, category, Search
├─ CENTER: results JPanel (expands)
└─ SOUTH: JPanel BorderLayout
   ├─ CENTER: JPanel FlowLayout – Prev, Next, Reserve
   └─ SOUTH: status JLabel
```
### 3.3 Screen implementation

In [2]:
%%writefile LibrarySearch.java
import javax.swing.*;
import java.awt.*;
public class LibrarySearch extends JFrame {
    private final JTextField query = new JTextField(22);
    private final JComboBox<String> category = new JComboBox<>(new String[]{"All", "Title", "Author", "Subject"});
    public LibrarySearch() {
        super("Campus Library Search"); setLayout(new BorderLayout(8,8));
        JPanel search = new JPanel(new FlowLayout(FlowLayout.LEFT));
        search.add(query); search.add(category); search.add(new JButton("Search")); add(search, BorderLayout.NORTH);
        JPanel results = new JPanel(new BorderLayout());
        results.add(new JLabel("Search results appear here", SwingConstants.CENTER), BorderLayout.CENTER);
        add(results, BorderLayout.CENTER);
        JPanel bottom = new JPanel(new BorderLayout()); JPanel buttons = new JPanel(new FlowLayout());
        buttons.add(new JButton("Prev")); buttons.add(new JButton("Next")); buttons.add(new JButton("Reserve"));
        bottom.add(buttons, BorderLayout.CENTER); bottom.add(new JLabel("Status: Ready"), BorderLayout.SOUTH);
        add(bottom, BorderLayout.SOUTH); setDefaultCloseOperation(EXIT_ON_CLOSE);
        setSize(700,450); setLocationRelativeTo(null);
    }
    public static void main(String[] args) { SwingUtilities.invokeLater(() -> new LibrarySearch().setVisible(true)); }
}


Writing LibrarySearch.java


In [3]:
!javac LibrarySearch.java
print("Compile succeeded. On desktop: java LibrarySearch")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Compile succeeded. On desktop: java LibrarySearch


### 3.4 Resizing evidence
Capture screenshots on desktop at small, normal and maximized sizes. The search and bottom rows should stay compact while the centre results panel expands; buttons/status should remain visible. Colab cannot display GUI screenshots.

### Alternative: GridBagLayout or absolute positioning
`GridBagLayout` provides flexible grid placement but its constraints are more complex. Absolute positioning (`setLayout(null)` and `setBounds`) often breaks with resize, font scaling or different platforms. Nested layout panels use **O(p)** container space for p panels; a flatter GridBag design can use fewer panels. Layout calculation is typically proportional to component/constraint count **O(c)**. Absolute coordinates are simple but poorly responsive.

## Task 4 – Event-Handling Forensics: Draw Pad (7 marks)
### 4.1 Defect
The callback is misspelled `mouseclicked`; the correct listener method is `mouseClicked(MouseEvent e)` (capital C). Without `@Override`, it is just a new method and never gets invoked. The label also is never updated.

### 4.2 Failing interaction
Click several times, then minimize and restore the window. Dots drawn with `getGraphics()` disappear because they were drawn on a transient screen surface and were not stored for repaint.

### 4.3 Repair
Store each clicked point, update label count, call `repaint()`, and draw all points in `paint()`. This preserves marks after repainting.

### 4.4 Prevention
`@Override` makes the compiler report a misspelled/nonmatching method signature before execution.

In [4]:
%%writefile DrawPad.java
import java.awt.*;
import java.awt.event.*;
import java.util.ArrayList;
import java.util.List;
public class DrawPad extends Frame {
    private final List<Point> points = new ArrayList<>();
    private final Label status = new Label("Clicks: 0");
    public DrawPad() {
        super("Draw Pad"); add(status, BorderLayout.SOUTH);
        addMouseListener(new MouseAdapter() {
            @Override public void mouseClicked(MouseEvent e) {
                points.add(e.getPoint()); status.setText("Clicks: " + points.size()); repaint();
            }
        });
        addWindowListener(new WindowAdapter() { @Override public void windowClosing(WindowEvent e) { dispose(); } });
        setSize(400,300);
    }
    @Override public void paint(Graphics g) {
        super.paint(g);
        for (Point p : points) g.fillOval(p.x-3, p.y-3, 6, 6);
    }
    public static void main(String[] args) { new DrawPad().setVisible(true); }
}


Writing DrawPad.java


In [5]:
!javac DrawPad.java
print("Compile succeeded. On desktop: java DrawPad")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Compile succeeded. On desktop: java DrawPad


### Complete-code typo experiment
Change `mouseClicked` to `mouseclicked` and remove `@Override`: it compiles as a separate method but the listener does not call it. If `@Override` remains, the compiler reports that it does not override a supertype method.

### Alternative: off-screen BufferedImage
For n points, repainting a point list takes **O(n)** time and point storage **O(n)** space. Drawing onto an off-screen image and copying it during repaint costs about **O(width×height)** pixels per repaint, independent of n; image memory is **O(width×height)**. Image rendering can be fast for many marks, but the image consumes fixed canvas memory.

## Task 5 – Currency Converter: Four Event Styles (7 marks)
### 5.1 Implementations
(a) Frame implements `ActionListener`; (b) named inner handler; (c) anonymous `ActionListener`; (d) lambda. A `WindowAdapter` handles closing. They can all call one shared `convert()` method.

### Lambda version – completed TODOs
Register one `ActionListener` lambda for both Convert and Enter in the amount field. Parse input, reject nonnumeric/negative values, format result to two decimals and show it in a dialog.

### 5.2 Requirement change
For Enter handling, add an action listener to the text field in each style. To show a dialog, change the shared conversion output method; sharing the method avoids duplicate code.

### 5.3 Comparison
| Style | Readability | Field access | Reuse | Maintenance |
|---|---|---|---|
| Frame implements | Easy for small class; class can grow cluttered | Direct | Low |
| Named inner class | Clear, reusable | Direct | Good |
| Anonymous inner | Local and explicit | Limited | Moderate |
| Lambda | Concise for one-method listener | Captures fields | Best for short handlers |
For larger applications, use named handlers/controller methods or shared `Action` objects for button, menu and shortcut triggers.

In [6]:
%%writefile Converter.java
import java.awt.FlowLayout;
import java.awt.event.*;
import javax.swing.*;
public class Converter extends JFrame {
    private final JTextField amount = new JTextField(10);
    private final JComboBox<String> target = new JComboBox<>(new String[]{"USD", "EUR"});
    private final JLabel result = new JLabel(" ");
    private static double rate(String currency) { return currency.equals("USD") ? 0.012 : 0.011; }
    public Converter() {
        super("Currency Converter"); setLayout(new FlowLayout()); JButton convertButton = new JButton("Convert");
        add(new JLabel("INR")); add(amount); add(target); add(convertButton); add(result);
        ActionListener action = e -> convert(); convertButton.addActionListener(action); amount.addActionListener(action);
        addWindowListener(new WindowAdapter(){ @Override public void windowClosing(WindowEvent e){ dispose(); } });
        setDefaultCloseOperation(DISPOSE_ON_CLOSE); pack(); setLocationRelativeTo(null);
    }
    private void convert() {
        try {
            double v = Double.parseDouble(amount.getText().trim());
            if (!Double.isFinite(v) || v < 0) throw new NumberFormatException();
            String cur = (String)target.getSelectedItem(); String msg = String.format("%.2f %s", v*rate(cur), cur);
            result.setText(msg); JOptionPane.showMessageDialog(this, msg, "Conversion", JOptionPane.INFORMATION_MESSAGE);
        } catch (NumberFormatException ex) {
            result.setText("Invalid amount"); JOptionPane.showMessageDialog(this, "Invalid amount", "Error", JOptionPane.ERROR_MESSAGE);
        }
    }
    public static void main(String[] args) { SwingUtilities.invokeLater(() -> new Converter().setVisible(true)); }
}


Writing Converter.java


In [7]:
!javac Converter.java
print("Compile succeeded. On desktop: java Converter")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Compile succeeded. On desktop: java Converter


### Other event styles (short code patterns)
**Frame implements:** `class ConverterFrame extends JFrame implements ActionListener` → register `this`; implement `actionPerformed` and call `convert()`.  
**Named inner class:** `class ConvertHandler implements ActionListener { public void actionPerformed(ActionEvent e){ convert(); } }` then register `new ConvertHandler()`.  
**Anonymous class:** `button.addActionListener(new ActionListener(){ public void actionPerformed(ActionEvent e){ convert(); } });`.

### Alternative: method reference or shared Action
`button.addActionListener(this::convert)` is compact. A shared Swing `Action` can be attached to button, menu and shortcut, centralizing behavior/state. One handler can serve multiple triggers and avoid duplicate logic; components still keep listener registrations.

## Task 6 – Campus Event Registration Desktop Application (13 marks)
### Stage 1: requirements
Users: students register; committee members view/filter/export. Screens: registration form, registrations table, event filter, export, confirmation/error dialogs. Data: student ID/name, event, timestamp. Testable requirements: reject blank fields; one registration per student/event; clearly report errors; keep GUI responsive during DB work; filter records; export current list.

### Stage 2: screen and flow
```text
JFrame + JMenuBar
├─ Registration form: ID, name, event, Register
├─ JTable: registrations, event filter, Refresh, Export
└─ Dialogs: validation, success, duplicate, DB error
Register → validate → background insert → duplicate/success result → refresh table
Filter → background paged query → update JTable
Export → write selected/filtered records
```

### Stage 3: decisions
Validate in UI and service. Enforce duplicate prevention with a database unique constraint on `(student_id,event_id)` (authoritative under concurrent requests). JDBC belongs in DAO. Use `SwingWorker` for queries/inserts/exports; update Swing components on EDT.

### Stage 4: layered class design
```text
EventRegistrationFrame (Swing UI/handlers)\n          ↓
RegistrationService (validation/business rules)
          ↓
RegistrationDAO (JDBC)
          ↓
MySQL
```
Register pseudocode: validate fields → if invalid show message → background task calls service/DAO → show duplicate/success/error → refresh table. Filter pseudocode: selected event/page → background `LIMIT/OFFSET` query → replace table model and hide loading indicator.

### Stage 5: implementation & demonstration
The sample code below is a GUI scaffold, plus a JDBC DAO example and SQL schema. Connect the UI actions to the service/DAO for the full database-backed workflow. Configure MySQL credentials and Connector/J. Demonstrate valid, invalid and duplicate cases.

### MySQL schema

In [9]:
# 1. Install the mysql-connector-python package (if not already installed)
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 43.7 MB/s  0:00:00


In [12]:
# 1. Install and start MySQL Server inside Colab
!apt-get update > /dev/null
!apt-get install -y mysql-server > /dev/null
!service mysql start

import mysql.connector

# 2. Establish connection to the local MySQL Server
# Default root password is empty when installing on Ubuntu/Debian via apt-get
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password=""
    )
    cursor = conn.cursor()

    # 3. Define the SQL script from your assignment schema
    sql_script = """
    CREATE DATABASE IF NOT EXISTS campus_events;
    USE campus_events;

    CREATE TABLE IF NOT EXISTS events (
        event_id INT PRIMARY KEY AUTO_INCREMENT,
        event_name VARCHAR(120) NOT NULL UNIQUE,
        event_date DATE
    );

    CREATE TABLE IF NOT EXISTS registrations (
        registration_id INT PRIMARY KEY AUTO_INCREMENT,
        student_id VARCHAR(30) NOT NULL,
        student_name VARCHAR(100) NOT NULL,
        event_id INT NOT NULL,
        registered_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        CONSTRAINT fk_event FOREIGN KEY(event_id) REFERENCES events(event_id),
        CONSTRAINT uq_student_event UNIQUE(student_id, event_id)
    );

    INSERT IGNORE INTO events(event_name, event_date) VALUES
    ('Tech Symposium', '2026-12-10'),
    ('Cultural Fest', '2026-12-15');
    """

    # 4. Execute the statements
    for result in cursor.execute(sql_script, multi=True):
        if result.with_rows:
            print("Rows produced:", result.fetchall())
        else:
            print(f"Affected {result.rowcount} rows")

    conn.commit()
    print("\nDatabase schema and tables initialized successfully inside Colab!")

except mysql.connector.Error as err:
    print(f"Error: {err}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Operation was interrupted before it could finish
^C
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
Error: 1698 (28000): Access denied for user 'root'@'localhost'


### JDBC DAO example (PreparedStatement)
Add MySQL Connector/J to your classpath and update the connection credentials. This example demonstrates insertion with parameters and a paged query.

In [13]:
%%writefile RegistrationDAO.java
import java.sql.*;
public class RegistrationDAO {
    private final String url="jdbc:mysql://localhost:3306/campus_events";
    private final String user="root", password="CHANGE_ME";
    public void register(String sid,String name,int eventId) throws SQLException {
        String sql="INSERT INTO registrations(student_id,student_name,event_id) VALUES(?,?,?)";
        try(Connection c=DriverManager.getConnection(url,user,password); PreparedStatement ps=c.prepareStatement(sql)) {
            ps.setString(1,sid); ps.setString(2,name); ps.setInt(3,eventId); ps.executeUpdate();
        }
    }
    public ResultSet pageForEvent(Connection c,int eventId,int limit,int offset) throws SQLException {
        String sql="SELECT registration_id,student_id,student_name,event_id,registered_at FROM registrations WHERE event_id=? ORDER BY registration_id LIMIT ? OFFSET ?";
        PreparedStatement ps=c.prepareStatement(sql); ps.setInt(1,eventId); ps.setInt(2,limit); ps.setInt(3,offset); return ps.executeQuery();
    }
}


Writing RegistrationDAO.java


### Swing UI scaffold

In [14]:
%%writefile EventRegistrationFrame.java
import javax.swing.*; import java.awt.*;
public class EventRegistrationFrame extends JFrame {
 private final JTextField sid=new JTextField(10), name=new JTextField(14);
 private final JComboBox<String> event=new JComboBox<>(new String[]{"Tech Symposium","Cultural Fest"});
 private final JTable table=new JTable(new Object[][]{},new String[]{"ID","Student","Event","Registered At"});
 public EventRegistrationFrame(){
  super("Campus Event Registration"); JMenuBar mb=new JMenuBar(); JMenu menu=new JMenu("File"); JMenuItem export=new JMenuItem("Export"), exit=new JMenuItem("Exit"); menu.add(export);menu.addSeparator();menu.add(exit);mb.add(menu);setJMenuBar(mb);
  JPanel form=new JPanel(new FlowLayout(FlowLayout.LEFT));form.add(new JLabel("Student ID"));form.add(sid);form.add(new JLabel("Name"));form.add(name);form.add(new JLabel("Event"));form.add(event);JButton reg=new JButton("Register");form.add(reg);
  JPanel bottom=new JPanel(new FlowLayout(FlowLayout.LEFT));bottom.add(new JLabel("Filter event:"));bottom.add(event);bottom.add(new JButton("Refresh"));
  add(form,BorderLayout.NORTH);add(new JScrollPane(table),BorderLayout.CENTER);add(bottom,BorderLayout.SOUTH);
  reg.addActionListener(e->JOptionPane.showMessageDialog(this,"Connect this action to RegistrationService/DAO."));exit.addActionListener(e->dispose());export.addActionListener(e->JOptionPane.showMessageDialog(this,"Connect export implementation."));
  setDefaultCloseOperation(DISPOSE_ON_CLOSE);setSize(800,500);setLocationRelativeTo(null);
 }
 public static void main(String[]args){SwingUtilities.invokeLater(()->new EventRegistrationFrame().setVisible(true));}
}


Writing EventRegistrationFrame.java


In [15]:
!javac EventRegistrationFrame.java
print("Compile succeeded. Run locally: java EventRegistrationFrame")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Compile succeeded. Run locally: java EventRegistrationFrame


### SwingWorker pattern for background work
Use `doInBackground()` for service/DAO calls and `done()` for UI feedback on the EDT. This pattern is illustrative and requires a variable named `frame` referencing your window.